# Raytracing ray image

`batcamp` also provides an octree-aware ray tracer. This notebook walks through one simple line-of-sight image: load one inner-heliosphere dataset, build the octree objects, define one camera plane of parallel rays, inspect how many cells each ray crosses, and finally integrate density along those rays.


## Load one dataset and build the raytracing objects

We start from one sample BATSRUS file that ships with the repository. The same reconstructed octree supports both interpolation and ray integration, so we build one `OctreeInterpolator` and one `OctreeRayTracer` from the same tree.


In [ ]:
from batread import Dataset
import matplotlib.pyplot as plt
import numpy as np

from batcamp import Octree, OctreeInterpolator, OctreeRayTracer

ds = Dataset.from_file("../sample_data/3d__var_2_n00060005.plt")
print(ds)

tree = Octree.from_ds(ds)
interp = OctreeInterpolator(tree, ds["Rho [g/cm^3]"])
tracer = OctreeRayTracer(tree)

print(tree)
print(interp)
print(tracer)


## Define one image plane of rays

Here the image plane is the square in the $(y, z)$ coordinates, while the rays all start at the same upstream $x$ position and travel in the $+x$ direction. The ray origins therefore form one `(ny, nz, 3)` array, while the direction is one shared 3-vector.


In [ ]:
coords = np.linspace(-215.0, 215.0, 512)
y, z = np.meshgrid(coords, coords, indexing="xy")
origins = np.stack((np.full_like(y, -400.0), y, z), axis=-1)
directions = np.array([1.0, 0.0, 0.0], dtype=float)

print(f"origin grid shape: {origins.shape}")
print(f"shared direction shape: {directions.shape}")
print(f"first ray origin: {origins[0, 0]}")
print(f"ray direction: {directions}")


## Trace the rays through the octree

The low-level `trace` call tells us which octree cells each ray crosses. A simple first diagnostic is the number of traversed cells per pixel: it shows where the domain is thicker or more refined along the line of sight.


In [ ]:
segments = tracer.trace(origins, directions)
cell_counts = np.diff(segments.ray_offsets).reshape(y.shape)

print(
    f"cells per ray: min={int(cell_counts.min())}, "
    f"median={float(np.median(cell_counts)):.1f}, "
    f"max={int(cell_counts.max())}"
)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
mesh = ax.pcolormesh(y, z, cell_counts, shading="auto")
ax.set_xlabel("y")
ax.set_ylabel("z")
ax.set_title("Traversed octree cells per ray")
fig.colorbar(mesh, ax=ax, label="cell count")
plt.show()


## Integrate density along the rays

`trilinear_image` evaluates the interpolated density along each ray and integrates it over the traversed path segments. The output has the same image-plane shape as the input ray grid, so it can be plotted directly as one 2D image.


In [ ]:
rho_los, _ = tracer.trilinear_image(interp, origins, directions)
positive = rho_los[np.isfinite(rho_los) & (rho_los > 0.0)]
print(f"image shape: {rho_los.shape}")
print(f"positive line integrals: min={positive.min():.3e}, max={positive.max():.3e}")

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
mesh = ax.pcolormesh(y, z, rho_los, shading="auto", norm="log")
ax.set_xlabel("y")
ax.set_ylabel("z")
ax.set_title("Cartesian ray-integrated density")
fig.colorbar(mesh, ax=ax, label="line integral of Rho [g/cm^3]")
plt.show()
